# 正式 probing：固定 240 train / 80 validation
一个持久 Engine，单次最多8小时；s=0一次，非零scale按3→5→7累计采样。
已就绪环境运行 cell 9；cell 10 在同一输出目录续跑。跨 Kaggle session 必须先恢复完整 probing_v1 目录，再运行；/kaggle/working 不会自动跨session保留。
遇到环境错误或 OOM 停止，不自动重试。完成前不发布最终训练题集。


In [ ]:
!pip install -q uv

In [ ]:
%%bash

uv python install 3.11
uv venv --python 3.11 --seed /kaggle/working/latent_sglang

In [ ]:
%%bash

/kaggle/working/latent_sglang/bin/python --version

In [ ]:
%%bash
git clone https://github.com/fengjixing-stupid/ECNU-Adaptive-Noise-Latent-GRPO.git
cd /kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final
git clone https://github.com/fengjixing-stupid/Latent-GRPO.git

In [ ]:
%%bash
cd /kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final/Latent-GRPO/Latent-GRPO/sglang_latent_reasoning_pkg
/kaggle/working/latent_sglang/bin/python -m pip install -U pip

/kaggle/working/latent_sglang/bin/python -m pip install -e "python[all]"

In [ ]:
%%bash

/kaggle/working/latent_sglang/bin/python - <<'PY'
import sys
import sglang

print("Python:", sys.version)
print("SGLang:", sglang.__version__)
PY

In [ ]:
%%bash

PY=/kaggle/working/latent_sglang/bin/python

$PY -m pip uninstall -y torchao compressed-tensors

$PY -m pip install \
    torchao==0.9.0 \
    compressed-tensors==0.11.0

In [ ]:
%%bash

/kaggle/working/latent_sglang/bin/python - <<'PY'
import torch
import torchao
import transformers
import compressed_tensors
import sglang

from importlib.metadata import version

print("torch:", torch.__version__)
print("torchao:", version("torchao"))
print("transformers:", transformers.__version__)
print("compressed-tensors:", version("compressed-tensors"))
print("sglang:", sglang.__version__)

from sglang.srt.entrypoints.engine import Engine
print("Engine import OK")
PY

In [ ]:
%%bash

/kaggle/working/latent_sglang/bin/python - <<'BUNDLE'
from pathlib import Path
files = {'adaptive_noise/__init__.py': '"""Small PyTorch components for frozen latent inference."""\n', 'adaptive_noise/dataset_builder.py': '"""Build a small, immutable candidate pool without loading a model."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport random\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom adaptive_noise.seeds import derive_dataset_seeds\n\nTRAIN_SOURCES = (\'gsm8k_aug\', \'dapo_math\')\nTEST_FILES = {\'gsm8k_aug_test\': \'test_gsm8k_aug.parquet\', \'math500_test\': \'test_math500.parquet\'}\nROW_SCHEMA = pa.schema([\n    (\'problem_id\', pa.string()), (\'source\', pa.string()), (\'source_file_sha256\', pa.string()),\n    (\'source_row\', pa.int64()), (\'original_index\', pa.int64()), (\'original_split\', pa.string()),\n    (\'split\', pa.string()), (\'question\', pa.string()),\n    (\'prompt\', pa.list_(pa.struct([(\'role\', pa.string()), (\'content\', pa.string())]))),\n    (\'ground_truth\', pa.string()), (\'reward_style\', pa.string())])\nREJECT_SCHEMA = pa.schema([(name, ROW_SCHEMA.field(name).type) for name in\n                          (\'problem_id\', \'source\', \'source_file_sha256\', \'source_row\')] + [(\'reason\', pa.string())])\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef iter_rows(path):\n    offset = 0\n    for batch in pq.ParquetFile(path).iter_batches(batch_size=2048):\n        for row in batch.to_pylist():\n            yield offset, row\n            offset += 1\n\n\ndef _normalize(row, source, digest, index):\n    info = row.get(\'extra_info\') or {}\n    reward = row.get(\'reward_model\') or {}\n    prompt = row.get(\'prompt\')\n    question = info.get(\'question\')\n    answer = reward.get(\'ground_truth\')\n    if not isinstance(question, str) or not question.strip():\n        raise ValueError(\'empty_question\')\n    if not isinstance(answer, str) or not answer.strip():\n        raise ValueError(\'empty_ground_truth\')\n    if not isinstance(prompt, list) or not prompt or not all(\n        isinstance(m, dict) and m.get(\'role\') in {\'user\', \'system\', \'assistant\'}\n        and isinstance(m.get(\'content\'), str) and m[\'content\'].strip() for m in prompt):\n        raise ValueError(\'invalid_prompt\')\n    if not any(m[\'role\'] == \'user\' for m in prompt):\n        raise ValueError(\'missing_user_message\')\n    return {\'problem_id\': f\'{source}:{digest}:{index}\', \'source\': source,\n            \'source_file_sha256\': digest, \'source_row\': index,\n            \'original_index\': info.get(\'index\'), \'original_split\': info.get(\'split\'),\n            \'split\': \'\', \'question\': question, \'prompt\': prompt,\n            \'ground_truth\': answer, \'reward_style\': reward.get(\'style\')}\n\n\ndef _write(path, rows, schema):\n    pq.write_table(pa.Table.from_pylist(rows, schema=schema), path)\n\n\ndef build_dataset(config, root):\n    root = Path(root)\n    sources = TRAIN_SOURCES + tuple(TEST_FILES)\n    paths = {}\n    for source in sources:\n        path = Path(config[\'inputs\'][source])\n        path = path if path.is_absolute() else root / path\n        if path.suffix != \'.parquet\' or not path.is_file():\n            raise FileNotFoundError(f\'missing required parquet input: {path}\')\n        paths[source] = path\n    seeds = derive_dataset_seeds(config[\'master_seed\'])\n    sample, train, val = (config[k] for k in (\'sample_per_source\', \'train_per_source\', \'validation_per_source\'))\n    if any(isinstance(n, bool) or not isinstance(n, int) or n <= 0 for n in (sample, train, val)) or train + val != sample:\n        raise ValueError(\'positive train and validation counts must sum to sample_per_source\')\n    output = Path(config[\'output_dir\'])\n    output = output if output.is_absolute() else root / output\n    if output.exists():\n        raise FileExistsError(f\'refusing to overwrite output: {output}\')\n    hashes = {source: file_sha256(path) for source, path in paths.items()}\n    rejected, train_rows, val_rows = [], [], []\n    input_stats = {}\n    split_rng = random.Random(seeds[\'train_validation_split\'])\n    for source, role in zip(TRAIN_SOURCES, (\'gsm8k_aug_sample\', \'dapo_sample\')):\n        valid = []\n        count = 0\n        for index, row in iter_rows(paths[source]):\n            count += 1\n            try:\n                _normalize(row, source, hashes[source], index)\n                valid.append(index)\n            except ValueError as error:\n                rejected.append({\'problem_id\': f\'{source}:{hashes[source]}:{index}\', \'source\': source,\n                                 \'source_file_sha256\': hashes[source], \'source_row\': index, \'reason\': str(error)})\n        if len(valid) < sample:\n            raise ValueError(f\'insufficient valid candidates for {source}: {len(valid)} < {sample}\')\n        selected = sorted(random.Random(seeds[role]).sample(valid, sample))\n        split_rng.shuffle(selected)\n        train_ids = set(selected[:train])\n        selected_ids = set(selected)\n        for index, row in iter_rows(paths[source]):\n            if index not in selected_ids:\n                continue\n            record = _normalize(row, source, hashes[source], index)\n            record[\'split\'] = \'train\' if index in train_ids else \'validation\'\n            (train_rows if index in train_ids else val_rows).append(record)\n        input_stats[source] = {\'raw\': count, \'valid\': len(valid), \'rejected\': count - len(valid)}\n    tests = {}\n    for source in TEST_FILES:\n        records = []\n        for index, row in iter_rows(paths[source]):\n            try:\n                record = _normalize(row, source, hashes[source], index)\n            except ValueError as error:\n                raise ValueError(f\'invalid test record {source}:{index}: {error}\') from error\n            record[\'split\'] = \'test\'\n            records.append(record)\n        tests[source] = records\n        input_stats[source] = {\'raw\': len(records), \'valid\': len(records), \'rejected\': 0}\n    # Recheck before publishing: all selected IDs refer to these input bytes.\n    if any(file_sha256(path) != hashes[source] for source, path in paths.items()):\n        raise ValueError(\'input changed during construction\')\n    output.mkdir(parents=True, exist_ok=False)\n    _write(output / \'train.parquet\', train_rows, ROW_SCHEMA)\n    _write(output / \'validation.parquet\', val_rows, ROW_SCHEMA)\n    for source, name in TEST_FILES.items():\n        _write(output / name, tests[source], ROW_SCHEMA)\n    all_rows = train_rows + val_rows + [r for rows in tests.values() for r in rows]\n    manifest_schema = pa.schema([(name, ROW_SCHEMA.field(name).type) for name in (\'problem_id\', \'source\', \'split\')])\n    _write(output / \'split_manifest.parquet\', all_rows, manifest_schema)\n    _write(output / \'rejected.parquet\', rejected, REJECT_SCHEMA)\n    counts = {\'train\': len(train_rows), \'validation\': len(val_rows),\n              **{source: len(rows) for source, rows in tests.items()}, \'rejected\': len(rejected)}\n    manifest = {\'schema_version\': 1, \'status\': \'complete\', \'master_seed\': config[\'master_seed\'],\n                \'derived_seeds\': seeds, \'seed_algorithm\': \'sha256-v1\',\n                \'rng_algorithm\': \'python.random.Random\', \'python\': platform.python_version(),\n                \'pyarrow\': pa.__version__, \'config\': config,\n                \'inputs\': {source: {\'path\': str(path), \'sha256\': hashes[source], **input_stats[source]}\n                           for source, path in paths.items()}, \'counts\': counts,\n                \'deduplication\': \'disabled_by_user\',\n                \'output_sha256\': {p.name: file_sha256(p) for p in sorted(output.glob(\'*.parquet\'))}}\n    temporary = output / \'build_manifest.json.tmp\'\n    temporary.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.rename(output / \'build_manifest.json\')\n    return manifest\n', 'adaptive_noise/seeds.py': '"""Stable seed identities for data construction and frozen-model probing."""\nimport hashlib\nimport json\nimport math\n\nDATA_ROLES = (\'gsm8k_aug_sample\', \'dapo_sample\', \'train_validation_split\')\n\n\ndef _uint32(data):\n    return int.from_bytes(hashlib.sha256(data).digest()[:4], \'big\')\n\n\ndef _integer(value, name):\n    if isinstance(value, bool) or not isinstance(value, int) or value < 0:\n        raise ValueError(f\'{name} must be a non-negative integer\')\n    return value\n\n\ndef derive_dataset_seeds(master_seed):\n    _integer(master_seed, \'master_seed\')\n    return {role: _uint32(f\'adanoise.dataset-seeds.v1:{master_seed}:{role}\'.encode(\'utf-8\'))\n            for role in DATA_ROLES}\n\n\ndef derive_rollout_seed(problem_id, scale, rollout_id, root_seed=12345):\n    _integer(root_seed, \'root_seed\')\n    _integer(rollout_id, \'rollout_id\')\n    if not isinstance(problem_id, str) or not problem_id:\n        raise ValueError(\'problem_id must be non-empty\')\n    if isinstance(scale, bool) or not math.isfinite(float(scale)) or float(scale) < 0:\n        raise ValueError(\'scale must be finite and non-negative\')\n    scale_string = format(float(scale) if float(scale) != 0 else 0.0, \'.17g\')\n    payload = json.dumps([root_seed, problem_id, scale_string, rollout_id],\n                         ensure_ascii=True, separators=(\',\', \':\')).encode(\'ascii\')\n    return _uint32(b\'adanoise.probe-seeds.v1:\' + payload)\n', 'adaptive_noise/probing.py': '"""Pure CPU control of cumulative frozen-checkpoint probing."""\nfrom fractions import Fraction\nimport math\n\nfrom adaptive_noise.seeds import derive_rollout_seed\n\nSCALES = (0.0, .25, .5, 1.0, 2.0)\nSOURCE_MAX_TOKENS = {\'gsm8k_aug\': 128, \'dapo_math\': 4096}\n\n\ndef decide_probe(rows, split):\n    if split not in {\'train\', \'validation\'}:\n        raise ValueError(\'probing selection accepts only train/validation\')\n    buckets = {s: {} for s in SCALES}\n    identities = set()\n    for row in rows:\n        scale, index, reward = row[\'scale\'], row[\'rollout_id\'], row[\'reward\']\n        if isinstance(scale, bool) or not math.isfinite(float(scale)) or float(scale) not in buckets:\n            raise ValueError(\'unknown scale in approved probing grid\')\n        scale = float(scale)\n        if isinstance(index, bool) or not isinstance(index, int) or index < 0 or index >= (1 if scale == 0 else 7):\n            raise ValueError(\'rollout_id exceeds approved budget\')\n        if row.get(\'status\') != \'completed\' or reward not in (0, 1) or reward is None:\n            raise ValueError(\'only completed binary-reward rollouts can be counted\')\n        if index in buckets[scale]:\n            raise ValueError(\'duplicate rollout identity\')\n        buckets[scale][index] = int(reward)\n        identities.add(row[\'problem_id\'])\n    if len(identities) > 1:\n        raise ValueError(\'decision rows must refer to one problem_id\')\n    for bucket in buckets.values():\n        if sorted(bucket) != list(range(len(bucket))):\n            raise ValueError(\'non-contiguous rollout IDs\')\n    counts = {s: len(buckets[s]) for s in SCALES}\n    summary = [{\'scale\': s, \'completed_count\': counts[s], \'success_count\': sum(buckets[s].values()),\n                \'q_hat\': sum(buckets[s].values()) / counts[s] if counts[s] else None} for s in SCALES]\n    last_complete = None\n    # Decisions only use complete rounds. A terminal round cannot have later rows.\n    for m in (3, 5, 7):\n        if counts[0] == 1 and all(counts[s] >= m for s in SCALES if s != 0):\n            rates = {s: Fraction(sum(buckets[s][i] for i in range(1 if s == 0 else m)), 1 if s == 0 else m) for s in SCALES}\n            delta = max(rates.values()) - min(rates.values())\n            reason = None\n            if all(q == 0 for q in rates.values()): reason = \'all_wrong\'\n            elif all(q == 1 for q in rates.values()): reason = \'all_correct\'\n            elif delta > Fraction(2, 5): reason = \'noise_sensitive\'\n            elif m == 7: reason = \'not_sensitive_at_cap\'\n            if reason:\n                if any(counts[s] > (1 if s == 0 else m) for s in SCALES):\n                    raise ValueError(\'rows collected after terminal decision\')\n                best = min(s for s in SCALES if rates[s] == max(rates.values()))\n                return {\'terminal\': True, \'reason\': reason, \'stage\': m, \'target_m\': m,\n                        \'keep\': split == \'validation\' or reason == \'noise_sensitive\',\n                        \'delta\': float(delta), \'s_star\': best, \'summary\': summary}\n            last_complete = {\'stage\': m, \'delta\': float(delta)}\n        else:\n            if any(counts[s] > m for s in SCALES if s != 0):\n                raise ValueError(\'later-stage rows before preceding round completed\')\n            return {\'terminal\': False, \'reason\': \'continue_probing\' if last_complete and all(counts[s] == last_complete[\'stage\'] for s in SCALES if s != 0) else \'incomplete\',\n                    \'stage\': last_complete[\'stage\'] if last_complete else None, \'target_m\': m,\n                    \'keep\': None, \'delta\': last_complete[\'delta\'] if last_complete else None,\n                    \'s_star\': None, \'summary\': summary}\n        # A completed non-terminal round advances to the next cumulative target.\n    raise AssertionError(\'unreachable probing state\')\n\n\ndef generation_settings(source, scale):\n    if source not in SOURCE_MAX_TOKENS:\n        raise ValueError(\'unknown training source\')\n    return {\'max_new_tokens\': SOURCE_MAX_TOKENS[source], \'temperature\': .6,\n            \'top_p\': .95, \'max_topk\': 10, \'gumbel_softmax_temperature\': 1.0,\n            \'noise_scale\': float(scale), \'add_noise_gumbel_softmax\': True,\n            \'use_one_sided_gumbel_noise\': False}\n\n\ndef plan_requests(problem_id, split, source, rows):\n    if source not in SOURCE_MAX_TOKENS:\n        raise ValueError(\'unknown training source\')\n    if any(row[\'problem_id\'] != problem_id for row in rows):\n        raise ValueError(\'foreign problem_id in rollout history\')\n    decision = decide_probe(rows, split)\n    if decision[\'terminal\']:\n        return []\n    existing = {(float(row[\'scale\']), row[\'rollout_id\']) for row in rows}\n    requests = []\n    for scale in SCALES:\n        for index in range(1 if scale == 0 else decision[\'target_m\']):\n            if (scale, index) in existing:\n                continue\n            requests.append({\'problem_id\': problem_id, \'source\': source, \'split\': split,\n                             \'scale\': scale, \'rollout_id\': index,\n                             \'seed\': derive_rollout_seed(problem_id, scale, index),\n                             **generation_settings(source, scale)})\n    return requests\n', 'adaptive_noise/probe_selection.py': '"""Validate CPU probing artifacts and retain the best-scale trajectories."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom adaptive_noise.dataset_builder import file_sha256, ROW_SCHEMA\nfrom adaptive_noise.probing import decide_probe, plan_requests, generation_settings\nfrom adaptive_noise.seeds import derive_rollout_seed\n\n\ndef _candidates(directory):\n    directory = Path(directory)\n    manifest_path = directory / \'build_manifest.json\'\n    manifest = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'status\') != \'complete\':\n        raise ValueError(\'candidate build is not complete\')\n    for name, digest in manifest[\'output_sha256\'].items():\n        path = directory / name\n        if not path.resolve().is_relative_to(directory.resolve()):\n            raise ValueError(\'invalid candidate artifact path\')\n        if not path.is_file():\n            raise FileNotFoundError(f\'missing required candidate artifact: {path}\')\n        if file_sha256(path) != digest:\n            raise ValueError(f\'candidate artifact changed: {path}\')\n    records = []\n    for split in (\'train\', \'validation\'):\n        for row in pq.read_table(directory / f\'{split}.parquet\').to_pylist():\n            if row[\'split\'] != split or row[\'source\'] not in {\'gsm8k_aug\',\'dapo_math\'}:\n                raise ValueError(\'candidate source/split mismatch\')\n            records.append(row)\n    if len({row[\'problem_id\'] for row in records}) != len(records):\n        raise ValueError(\'duplicate candidate IDs\')\n    mapping = {row[\'problem_id\']: row for row in records}\n    return manifest, mapping\n\n\ndef _group(mapping, rows):\n    groups = {pid: [] for pid in mapping}\n    for row in rows:\n        pid = row[\'problem_id\']\n        if pid not in mapping:\n            raise ValueError(f\'unknown candidate problem_id: {pid}\')\n        candidate = mapping[pid]\n        if row[\'source\'] != candidate[\'source\'] or row[\'split\'] != candidate[\'split\']:\n            raise ValueError(\'rollout source/split differs from fixed candidate split\')\n        if row[\'seed\'] != derive_rollout_seed(pid, row[\'scale\'], row[\'rollout_id\']):\n            raise ValueError(\'rollout seed differs from approved identity\')\n        groups[pid].append(row)\n    return groups\n\n\ndef plan_candidate_requests(candidate_dir, rows):\n    _, mapping = _candidates(candidate_dir)\n    groups = _group(mapping, rows)\n    requests = []\n    for pid, candidate in mapping.items():\n        requests.extend(plan_requests(pid, candidate[\'split\'], candidate[\'source\'], groups[pid]))\n    return requests\n\n\ndef _artifact(root, name):\n    if not isinstance(name, str) or not name:\n        raise FileNotFoundError(\'missing best-scale trajectory/features reference\')\n    path = (Path(root) / name).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError(\'trajectory/features path must remain inside trajectory_root\')\n    if path.suffix != \'.json\' or not path.is_file():\n        raise FileNotFoundError(f\'missing required trajectory/features JSON: {path}\')\n    return path\n\n\ndef _vector(values, *, probabilities=False, size=None):\n    if not isinstance(values, list) or not values or (size is not None and len(values) != size):\n        raise ValueError(\'missing or incorrect feature/mixture vector size\')\n    if any(isinstance(v,bool) or not isinstance(v,(float,int)) or not math.isfinite(v) for v in values):\n        raise ValueError(\'nonfinite or nonnumeric trajectory/feature values\')\n    if probabilities and (any(v < 0 for v in values) or abs(sum(values)-1) > 1e-5):\n        raise ValueError(\'probability vector must be nonnegative and sum to one\')\n\n\ndef _validate_pair(root, row):\n    paths = [_artifact(root, row.get(key)) for key in (\'trajectory_path\',\'features_path\')]\n    payloads = [json.loads(path.read_text(encoding=\'utf-8\')) for path in paths]\n    identities = []\n    for payload in payloads:\n        if payload.get(\'schema_version\') != 1 or any(payload.get(k) != row[k] for k in (\'problem_id\',\'scale\',\'rollout_id\',\'seed\')):\n            raise ValueError(\'trajectory/features identity mismatch\')\n        if payload.get(\'generation_config\') != generation_settings(row[\'source\'],row[\'scale\']):\n            raise ValueError(\'trajectory/features generation config differs from approved request\')\n        model = payload.get(\'model_identity\')\n        if not isinstance(model,str) or not model:\n            raise ValueError(\'missing model_identity\')\n        identities.append(model)\n    if identities[0] != identities[1]:\n        raise ValueError(\'trajectory and features model identity differs\')\n    trajectory, features = payloads\n    if trajectory.get(\'reward\') != row[\'reward\'] or not isinstance(trajectory.get(\'output_text\'),str):\n        raise ValueError(\'trajectory reward/output mismatch\')\n    tokens = trajectory.get(\'explicit_token_ids\')\n    if not isinstance(tokens,list) or any(isinstance(n,bool) or not isinstance(n,int) or n < 0 for n in tokens):\n        raise ValueError(\'missing/invalid explicit token sequence\')\n    steps, feature_steps = trajectory.get(\'steps\'), features.get(\'steps\')\n    if not isinstance(steps,list) or not steps or not isinstance(feature_steps,list) or len(steps)!=len(feature_steps):\n        raise ValueError(\'missing or misaligned latent trajectory/features\')\n    if features.get(\'feature_schema\')!=\'raw-latent-v1\' or features.get(\'feature_source\')!=\'pre_gumbel\':\n        raise ValueError(\'unsupported or noisy feature schema\')\n    hidden_size = None\n    for index,(step,feature) in enumerate(zip(steps,feature_steps)):\n        if step.get(\'step_idx\')!=index or feature.get(\'step_idx\')!=index:\n            raise ValueError(\'latent step alignment mismatch\')\n        ids = step.get(\'topk_ids\')\n        if not isinstance(ids,list) or len(ids)!=10 or any(isinstance(n,bool) or not isinstance(n,int) or n < 0 for n in ids):\n            raise ValueError(\'missing sparse top10 mixture IDs\')\n        _vector(step.get(\'mixture_probs\'), probabilities=True, size=10)\n        _vector(feature.get(\'hidden_state\'), size=hidden_size)\n        hidden_size = len(feature[\'hidden_state\'])\n        _vector(feature.get(\'topk_probs\'), probabilities=True, size=10)\n    return paths, [file_sha256(path) for path in paths], identities[0], hidden_size\n\n\ndef _table_write(path, rows, schema=None):\n    pq.write_table(pa.Table.from_pylist(rows, schema=schema), path)\n\n\ndef select_probe_dataset(candidate_dir, rollouts_path, output_dir, trajectory_root):\n    output = Path(output_dir)\n    if output.exists():\n        raise FileExistsError(f\'refusing to overwrite output: {output}\')\n    candidate_manifest, mapping = _candidates(candidate_dir)\n    rollouts_path = Path(rollouts_path)\n    if not rollouts_path.is_file():\n        raise FileNotFoundError(f\'missing required rollouts: {rollouts_path}\')\n    rows = pq.read_table(rollouts_path).to_pylist()\n    groups = _group(mapping, rows)\n    decisions, summaries, pairs = [], [], []\n    selected, validation = [], []\n    models, dimensions = set(), set()\n    for pid, candidate in mapping.items():\n        decision = decide_probe(groups[pid], candidate[\'split\'])\n        if not decision[\'terminal\']:\n            raise ValueError(f\'incomplete probing: {pid}; target_m={decision["target_m"]}\')\n        record = {k: candidate[k] for k in (\'problem_id\',\'source\',\'source_row\',\'source_file_sha256\',\'split\')}\n        record.update({k: decision[k] for k in (\'reason\',\'keep\',\'delta\',\'s_star\',\'stage\')})\n        decisions.append(record)\n        for summary in decision[\'summary\']:\n            scale_rows = [r for r in groups[pid] if r[\'scale\']==summary[\'scale\']]\n            summaries.append(dict(summary, problem_id=pid,split=candidate[\'split\'],source=candidate[\'source\'],\n                invalid_count=sum(r.get(\'invalid_reason\') is not None for r in scale_rows),\n                reason=decision[\'reason\'],stage=decision[\'stage\'],delta=decision[\'delta\']))\n        for row in groups[pid]:\n            if row[\'scale\'] != decision[\'s_star\']:\n                continue\n            paths, hashes, model, hidden_size = _validate_pair(trajectory_root,row)\n            models.add(model); dimensions.add(hidden_size)\n            pairs.append((row,paths,hashes))\n        if candidate[\'split\']==\'train\' and decision[\'keep\']:\n            selected.append(dict(candidate, noise_target=decision[\'s_star\']))\n        elif candidate[\'split\']==\'validation\':\n            validation.append(dict(candidate, probe_reason=decision[\'reason\'],noise_target=decision[\'s_star\']))\n    if len(models)!=1 or len(dimensions)!=1:\n        raise ValueError(\'mixed/missing model identity or hidden feature size\')\n    output.mkdir(parents=True,exist_ok=False)\n    (output/\'trajectories\').mkdir()\n    retained = []\n    for row, paths, hashes in pairs:\n        key = hashlib.sha256(json.dumps([row[\'problem_id\'],row[\'scale\'],row[\'rollout_id\']]).encode()).hexdigest()\n        record = {k: row[k] for k in (\'problem_id\',\'source\',\'split\',\'scale\',\'rollout_id\',\'seed\',\'reward\')}\n        for field,path,digest in zip((\'trajectory_path\',\'features_path\'),paths,hashes):\n            relative = f\'trajectories/{key}_{field}.json\'\n            shutil.copyfile(path,output/relative)\n            if file_sha256(output/relative)!=digest:\n                raise ValueError(\'trajectory/features input changed while copying\')\n            record[field]=relative\n        retained.append(record)\n    train_schema = pa.schema(list(ROW_SCHEMA)+[pa.field(\'noise_target\',pa.float64())])\n    val_schema = pa.schema(list(train_schema)+[pa.field(\'probe_reason\',pa.string())])\n    _table_write(output/\'train_selected.parquet\',selected,train_schema)\n    _table_write(output/\'validation_marked.parquet\',validation,val_schema)\n    _table_write(output/\'selection_decisions.parquet\',decisions)\n    _table_write(output/\'probe_summary.parquet\',summaries)\n    light_fields = (\'problem_id\',\'source\',\'split\',\'scale\',\'rollout_id\',\'seed\',\'status\',\'reward\',\n                    \'invalid_reason\',\'truncated\',\'finish_reason\',\'extracted_answer\')\n    _table_write(output/\'probe_rollouts.parquet\',[{k:r.get(k) for k in light_fields} for r in rows])\n    _table_write(output/\'retained_trajectories.parquet\',retained)\n    manifest = {\'schema_version\':1,\'status\':\'complete\',\'candidate_manifest_sha256\':file_sha256(Path(candidate_dir)/\'build_manifest.json\'),\n                \'rollouts_sha256\':file_sha256(rollouts_path),\'model_identity\':next(iter(models)),\n                \'feature_schema\':\'raw-latent-v1\',\'hidden_size\':next(iter(dimensions)),\n                \'counts\':{\'train_selected\':len(selected),\'validation\':len(validation),\n                          \'rollouts\':len(rows),\'retained_trajectories\':len(retained)},\n                \'scales\':[0,.25,.5,1,2],\'root_seed\':12345,\'seed_algorithm\':\'probe-sha256-v1\',\n                \'scope\':\'offline_selection_no_model_execution\',\n                \'output_sha256\':{p.name:file_sha256(p) for p in sorted(output.glob(\'*.parquet\'))}}\n    temp=output/\'selection_manifest.json.tmp\'\n    temp.write_text(json.dumps(manifest,indent=2)+\'\\n\',encoding=\'utf-8\')\n    temp.rename(output/\'selection_manifest.json\')\n    return manifest\n', 'adaptive_noise/answer_verifier.py': '"""Run only the pinned author\'s pure scoring helpers, without GPU imports."""\nimport ast\nfrom functools import lru_cache\nfrom pathlib import Path\nimport re\nimport subprocess\nimport warnings\n\nROOT = Path(__file__).resolve().parents[1]\nUPSTREAM = ROOT / \'latent_grpo_minfix/Latent-GRPO-final\'\nPINNED_COMMIT = \'0b7e85f15e9859033653964282348e518f9f6291\'\nCOMMON = (\'fix_fracs\', \'fix_a_slash_b\', \'remove_right_units\', \'fix_sqrt\', \'strip_string\',\n          \'remove_boxed\', \'last_boxed_only_string\')\nNAMES = {\'low\': COMMON + (\'extract_answer\', \'normalize_answer_text\', \'_as_float\', \'check_is_correct\'),\n         \'high\': COMMON + (\'is_equiv\', \'compute_score\')}\n\n\n@lru_cache(maxsize=4)\ndef _compile(profile, code, path):\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\', SyntaxWarning)  # pinned author\'s \\\\% string literal\n        tree = ast.parse(code, filename=path)\n    functions = {node.name: node for node in tree.body if isinstance(node, ast.FunctionDef)}\n    missing = set(NAMES[profile]) - functions.keys()\n    if missing:\n        raise ValueError(f\'missing required author helpers: {sorted(missing)}\')\n    namespace = {\'re\': re}\n    selected = ast.Module(body=[functions[name] for name in NAMES[profile]], type_ignores=[])\n    exec(compile(selected, path, \'exec\'), namespace)\n    return namespace\n\n\ndef load_author_helpers(profile, path=None):\n    if profile not in NAMES:\n        raise ValueError(\'unknown scoring profile\')\n    if path is None:\n        relative = f\'Latent-GRPO/eval/eval_{profile}_tasks_sglang.py\'\n        path = UPSTREAM / relative\n        if not path.is_file():\n            raise FileNotFoundError(f\'missing required reference: {path}\')\n        head = subprocess.check_output([\'git\', \'-C\', str(UPSTREAM), \'rev-parse\', \'HEAD\'], text=True).strip()\n        if head != PINNED_COMMIT:\n            raise ValueError(\'author reference HEAD differs from pinned commit\')\n        changed = subprocess.run([\'git\', \'-C\', str(UPSTREAM), \'diff\', \'--quiet\', \'HEAD\', \'--\', relative])\n        if changed.returncode:\n            raise ValueError(\'author scoring reference differs from pinned commit\')\n    path = Path(path)\n    return _compile(profile, path.read_text(encoding=\'utf-8\'), str(path))\n\n\ndef verify_answer(source, text, ground_truth, finish_reason=None):\n    profiles = {\'gsm8k_aug\': \'low\', \'gsm8k_aug_test\': \'low\', \'dapo_math\': \'high\', \'math500_test\': \'high\'}\n    if source not in profiles or not isinstance(text, str) or not isinstance(ground_truth, str) or not ground_truth.strip():\n        raise ValueError(\'invalid source, generated text or ground truth\')\n    profile = profiles[source]\n    helpers = load_author_helpers(profile)\n    answer = \'\'\n    if text.strip():\n        if profile == \'low\':\n            answer = helpers[\'extract_answer\'](text)\n        else:\n            boxed = helpers[\'last_boxed_only_string\'](text)\n            if boxed is not None:\n                try:\n                    answer = helpers[\'remove_boxed\'](boxed)\n                except AssertionError:\n                    answer = \'\'  # malformed model answer, not a scorer execution failure\n    invalid = \'empty_output\' if not text.strip() else (\'answer_unextractable\' if not answer.strip() else None)\n    reward = 0\n    if invalid is None:\n        reward = int(helpers[\'check_is_correct\'](answer, ground_truth) if profile == \'low\'\n                     else helpers[\'is_equiv\'](answer, ground_truth))\n    return {\'reward\': reward, \'invalid_reason\': invalid,\n            \'truncated\': finish_reason in (\'length\', \'max_tokens\') if finish_reason is not None else None,\n            \'finish_reason\': finish_reason, \'extracted_answer\': answer}\n', 'adaptive_noise/kaggle_probing.py': '"""Durable collection of approved fixed-pool probing; no model imports."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport uuid\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nfrom adaptive_noise.dataset_builder import ROW_SCHEMA, _normalize, file_sha256, iter_rows\nfrom adaptive_noise.probe_selection import _candidates, _group, _validate_pair, select_probe_dataset\nfrom adaptive_noise.probing import decide_probe, plan_requests\n\n\ndef atomic_json(path, payload):\n    path = Path(path)\n    temp = path.with_name(path.name + \'.tmp\')\n    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False)+\'\\n\')\n    temp.replace(path)\n\n\ndef request_key(row):\n    return hashlib.sha256(json.dumps([row[\'problem_id\'],float(row[\'scale\']),row[\'rollout_id\']],\n                                    separators=(\',\',\':\')).encode()).hexdigest()\n\n\ndef load_pool(directory):\n    return _candidates(directory)[1]\n\n\ndef make_locked_pool(directory):\n    manifest, mapping = _candidates(directory)\n    return {\'schema_version\':1, \'master_seed\':42,\n            \'original_manifest_sha256\':file_sha256(Path(directory)/\'build_manifest.json\'),\n            \'inputs\':{s:{\'sha256\':manifest[\'inputs\'][s][\'sha256\']} for s in (\'gsm8k_aug\',\'dapo_math\')},\n            \'rows\':[{k:r[k] for k in (\'source\',\'source_row\',\'split\')} for r in mapping.values()]}\n\n\ndef restore_candidates(locked, paths, output):\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(output)\n    if locked[\'schema_version\'] != 1 or locked[\'master_seed\'] != 42:\n        raise ValueError(\'unknown locked pool protocol\')\n    digests = {}\n    for source in (\'gsm8k_aug\',\'dapo_math\'):\n        digests[source] = file_sha256(paths[source])\n        if digests[source] != locked[\'inputs\'][source][\'sha256\']:\n            raise ValueError(f\'input identity mismatch: {source}\')\n    selected = {}\n    for r in locked[\'rows\']:\n        key = (r[\'source\'],r[\'source_row\'])\n        if key in selected or r[\'split\'] not in (\'train\',\'validation\') or r[\'source\'] not in digests:\n            raise ValueError(\'invalid locked row/split\')\n        if type(r[\'source_row\']) is not int or r[\'source_row\'] < 0:\n            raise ValueError(\'invalid locked source_row\')\n        selected[key] = r[\'split\']\n    if len(selected)!=320 or any(sum(s==source and split==part for (s,_),split in selected.items())!=count\n        for source in digests for part,count in [(\'train\',120),(\'validation\',40)]):\n        raise ValueError(\'locked pool must retain approved 120/40 per source\')\n    found = {}\n    for source in digests:\n        for index,row in iter_rows(paths[source]):\n            key=(source,index)\n            if key in selected:\n                normalized=_normalize(row,source,digests[source],index)\n                normalized[\'split\']=selected[key]\n                found[key]=normalized\n    if found.keys()!=selected.keys():\n        raise ValueError(\'missing locked source row\')\n    # Preserve the local fixed-pool order, independent of Python/Kaggle RNG version.\n    records=[found[(r[\'source\'],r[\'source_row\'])] for r in locked[\'rows\']]\n    if any(file_sha256(paths[s])!=digests[s] for s in digests):\n        raise ValueError(\'input changed while restoring pool\')\n    final=output\n    output=final.with_name(final.name+\'.pending-\'+uuid.uuid4().hex)\n    output.mkdir(parents=True)\n    for split in (\'train\',\'validation\'):\n        pq.write_table(pa.Table.from_pylist([r for r in records if r[\'split\']==split],schema=ROW_SCHEMA),output/f\'{split}.parquet\')\n    atomic_json(output/\'build_manifest.json\',dict(status=\'complete\',scope=\'locked_train_validation_restore_no_test_read\',\n        original_manifest_sha256=locked[\'original_manifest_sha256\'],master_seed=42,\n        inputs=locked[\'inputs\'],counts={\'train\':240,\'validation\':80},\n        output_sha256={p.name:file_sha256(p) for p in output.glob(\'*.parquet\')}))\n    output.rename(final)\n\n\nclass ProbeStore:\n    def __init__(self, root, candidates, model_identity):\n        self.root, self.candidates = Path(root), Path(candidates)\n        self.mapping=load_pool(candidates)\n        self.model_identity=model_identity\n        self.root.mkdir(parents=True,exist_ok=True)\n        identity={\'candidate_manifest_sha256\':file_sha256(self.candidates/\'build_manifest.json\'),\n                  \'model_identity\':model_identity,\'protocol\':\'adanoise.fixed-probing.v1\',\n                  \'scales\':[0,.25,.5,1,2],\'prefix_cache_enabled\':False,\'max_running_requests\':1}\n        path=self.root/\'collection_identity.json\'\n        if path.exists():\n            if json.loads(path.read_text())!=identity:\n                raise ValueError(\'collection candidate/model identity mismatch\')\n        else:\n            if (self.root/\'rollouts\').exists():\n                raise ValueError(\'collection identity missing for existing rollouts\')\n            atomic_json(path,identity)\n        for name in (\'rollouts\',\'attempts\'):\n            (self.root/name).mkdir(exist_ok=True)\n        self.history=[]\n        for path in sorted((self.root/\'rollouts\').glob(\'*/completion.json\')):\n            row=json.loads(path.read_text())\n            if request_key(row)!=path.parent.name or row.get(\'model_identity\')!=model_identity:\n                raise ValueError(\'completed rollout identity/model mismatch\')\n            self.history.append(row)\n            if (path.parent/\'events\').exists():\n                shutil.rmtree(path.parent/\'events\')  # finish cleanup of own atomically committed request\n        self.groups=_group(self.mapping,self.history)\n        self.decisions={}\n        for candidate in self.mapping.values():\n            decide_probe(self.rows(candidate),candidate[\'split\'])\n        self.finish_questions()\n\n    def rows(self,candidate):\n        return self.groups[candidate[\'problem_id\']]\n\n    def pending(self,candidate):\n        return plan_requests(candidate[\'problem_id\'],candidate[\'split\'],candidate[\'source\'],self.rows(candidate))\n\n    def new_attempt(self):\n        directory=self.root/\'attempts\'/uuid.uuid4().hex\n        (directory/\'events\').mkdir(parents=True)\n        return directory\n\n    def commit(self,row,attempt):\n        attempt=Path(attempt).resolve()\n        if attempt.parent != (self.root/\'attempts\').resolve():\n            raise ValueError(\'attempt outside this collection\')\n        key=request_key(row)\n        target=self.root/\'rollouts\'/key\n        if target.exists():\n            raise ValueError(\'rollout already completed\')\n        candidate=self.mapping[row[\'problem_id\']]\n        if not any(request_key(req)==key for req in self.pending(candidate)):\n            raise ValueError(\'unexpected/terminal rollout request\')\n        completed=dict(row,status=\'completed\',model_identity=self.model_identity,\n                       trajectory_path=\'trajectory.json\',features_path=\'features.json\')\n        _group(self.mapping,[completed])\n        _,_,model,_=_validate_pair(attempt,completed)\n        if model!=self.model_identity:\n            raise ValueError(\'collected model identity mismatch\')\n        completed.update(trajectory_path=f\'rollouts/{key}/trajectory.json\',features_path=f\'rollouts/{key}/features.json\')\n        atomic_json(attempt/\'completion.json\',completed)\n        # Atomic directory rename publishes both features and result together.\n        attempt.rename(target)\n        self.history.append(completed)\n        self.groups[completed[\'problem_id\']].append(completed)\n        shutil.rmtree(target/\'events\')  # own transient per-token stream; never source data\n        return completed\n\n    def finish_questions(self):\n        decisions=self.decisions\n        for candidate in self.mapping.values():\n            if candidate[\'problem_id\'] in decisions:\n                continue\n            rows=self.rows(candidate)\n            decision=decide_probe(rows,candidate[\'split\'])\n            if not decision[\'terminal\']:\n                continue\n            decisions[candidate[\'problem_id\']]=decision\n            for row in rows:\n                if row[\'scale\']==decision[\'s_star\']:\n                    _validate_pair(self.root,row)\n                else:\n                    for field in (\'trajectory_path\',\'features_path\'):\n                        path=(self.root/row[field]).resolve()\n                        expected=(self.root/\'rollouts\'/request_key(row)/Path(row[field]).name).resolve()\n                        if path!=expected or Path(row[field]).name not in (\'features.json\',\'trajectory.json\'):\n                            raise ValueError(\'invalid generated artifact path\')\n                        path.unlink(missing_ok=True)\n                    (self.root/\'rollouts\'/request_key(row)/\'prompt.json\').unlink(missing_ok=True)\n        atomic_json(self.root/\'decisions.json\',decisions)\n        return decisions\n\n    def snapshot(self,status):\n        path=self.root/\'probe_rollouts.parquet\'\n        # Nulls remain typed until subsequent records arrive; write only nonempty history.\n        if self.history:\n            temp=path.with_suffix(\'.parquet.tmp\')\n            pq.write_table(pa.Table.from_pylist(self.history),temp)\n            temp.replace(path)\n        groups={}\n        for row in self.history:\n            group=groups.setdefault((row[\'source\'],row[\'scale\']),{\'count\':0,\'success\':0,\'invalid\':0,\'generation_sec\':0.})\n            group[\'count\']+=1; group[\'success\']+=row[\'reward\']\n            group[\'invalid\']+=row.get(\'invalid_reason\') is not None\n            group[\'generation_sec\']+=row.get(\'instrumented_generation_sec\',0.)\n        report={\'status\':status,\'completed_rollouts\':len(self.history),\n                \'terminal_questions\':len(self.finish_questions()),\'total_questions\':len(self.mapping),\n                \'by_source_scale\':[dict(source=s,scale=scale,**v,q_hat=v[\'success\']/v[\'count\'],\n                    invalid_rate=v[\'invalid\']/v[\'count\'],mean_generation_sec=v[\'generation_sec\']/v[\'count\'])\n                    for (s,scale),v in sorted(groups.items())]}\n        atomic_json(self.root/\'collection_status.json\',report)\n        return report\n\n    def publish(self):\n        if any(not decide_probe(self.rows(c),c[\'split\'])[\'terminal\'] for c in self.mapping.values()):\n            raise ValueError(\'incomplete probing; no final selection published\')\n        self.snapshot(\'complete\')\n        out=self.root/\'selected\'\n        if (out/\'selection_manifest.json\').is_file():\n            return json.loads((out/\'selection_manifest.json\').read_text())\n        staging=self.root/\'exports\'\n        staging.mkdir(exist_ok=True)\n        pending=staging/uuid.uuid4().hex\n        manifest=select_probe_dataset(self.candidates,self.root/\'probe_rollouts.parquet\',pending,self.root)\n        pending.rename(out)\n        return manifest\n', 'scripts/kaggle_request_seed_smoke.py': '"""Single-GPU, persistent Engine, sequential per-request seed diagnostic."""\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\n\nMODEL_PATH = \'/kaggle/input/models/fengjixing/llama3-2-1b-instruct-latent-grpo-top10/other/default/1/LLaMA3.2-1B-Instruct-Latent-GRPO-Top10\'\nDATA_PATH = \'/kaggle/input/datasets/fengjixing/datasets-for-latent-grpo/data/GSM8k-Aug-oss-dup-all.parquet\'\nSOURCE_SHA256 = \'3766e3a83cd82ddd686392d8bc6ef6f262821490a09b694b2caed44f1a482501\'\nCANDIDATE_ROW = 7335\nPROBLEM_ID = \'gsm8k_aug:3766e3a83cd82ddd686392d8bc6ef6f262821490a09b694b2caed44f1a482501:7335\'\nGENERATION_CONFIG = {\'temperature\': 0.6, \'top_p\': 0.95, \'max_new_tokens\': 128, \'gumbel_softmax_temperature\': 1.0, \'noise_scale\': 1.0, \'add_noise_gumbel_softmax\': True, \'use_one_sided_gumbel_noise\': False}\nEFFECTIVE_GENERATION_CONFIG = {\'temperature\': 0.6, \'top_p\': 0.95, \'max_new_tokens\': 128, \'gumbel_softmax_temperature\': 1.0, \'noise_scale\': 1.0, \'add_noise_gumbel_softmax\': True, \'use_one_sided_gumbel_noise\': False, \'max_topk\': 10}\nENGINE_STARTUP_SEED = 12345\nSETUP_SOURCE_SHA256 = {\'1\': \'d61c544c62cb5c88925ff781b7044e92e36af4cd968ec9016e93a6dc45f90f52\', \'2\': \'3ebdc13202223903b48d68ea04669dceb1ca699043be75013981b5e561852d1f\', \'3\': \'2e54626cfa084cb4c367d9a56ab24b8f009c70e3e4000329dc21eb03768aff2d\', \'4\': \'4d8ce5ec73780379557e5434a011829cf89772054e7f61bf90672de941171b30\', \'5\': \'0a77d759020a3d2cfe78e92f53274b4e17aeb77d5f5970a76aa95a3eb69dfdb4\', \'6\': \'f393104aa30d9ec98e995f8f6ceadb412c50f6604b3d55fb5654ace59a9d4363\', \'7\': \'a2329ed7e98bc8b459634eea0794a14b69380727e8c0eae34ab0e76b6cc96eca\', \'8\': \'b7076b5cff4956a1da2bb1d9ebd6c24dd2abbe5e9fe7201ededd4830fc865011\'}\n\nBATCH_SOURCE = "def install_batch(batch_class):\\n    original_from = batch_class.from_schedule_batch.__func__\\n    original_filter = batch_class.filter_batch\\n    original_merge = batch_class.merge_batch\\n\\n    @classmethod\\n    def from_batch(cls, batch, vocab_size):\\n        result = original_from(cls, batch, vocab_size)\\n        metadata = []\\n        for req in batch.reqs:\\n            custom = req.sampling_params.custom_params or {}\\n            smoke = custom.get(\'adanoise_smoke\')\\n            metadata.append(dict(smoke, request_id=req.rid) if smoke is not None else None)\\n        result.adanoise_requests = metadata\\n        return result\\n\\n    def filtered(self, keep_indices, keep_indices_device):\\n        metadata = getattr(self, \'adanoise_requests\', None)\\n        result = original_filter(self, keep_indices, keep_indices_device)\\n        if metadata is not None:\\n            self.adanoise_requests = [metadata[i] for i in keep_indices]\\n        return result\\n\\n    def merged(self, other):\\n        lhs = getattr(self, \'adanoise_requests\', None)\\n        rhs = getattr(other, \'adanoise_requests\', None)\\n        if lhs is None or rhs is None:\\n            raise ValueError(\'request metadata missing during batch merge\')\\n        combined = list(lhs) + list(rhs)\\n        result = original_merge(self, other)\\n        self.adanoise_requests = combined\\n        return result\\n\\n    batch_class.from_schedule_batch = from_batch\\n    batch_class.filter_batch = filtered\\n    batch_class.merge_batch = merged\\n"\n\nCAPTURE_SOURCE = "\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport torch\\n\\ndef install(sampler_class):\\n    original = sampler_class.forward\\n    def captured(self, output, info, *args, enable_latent=False, **kwargs):\\n        root = os.environ.get(\'ADANOISE_SMOKE_ROOT\')\\n        metadata = getattr(info, \'adanoise_requests\', None)\\n        if not root or not enable_latent or not metadata or all(m is None for m in metadata):\\n            return original(self, output, info, *args, enable_latent=enable_latent, **kwargs)\\n        if len(metadata) != 1 or metadata[0] is None:\\n            raise ValueError(\'request seed smoke requires one active request\')\\n        request = metadata[0]\\n        rid, run_id, seed = request[\'request_id\'], request[\'run_id\'], request[\'seed\']\\n        if run_id not in (\'same_a\', \'same_b\', \'different\') or not isinstance(rid, str) or not rid:\\n            raise ValueError(\'invalid smoke request identity\')\\n        if type(seed) is not int or not 0 <= seed < 2**32:\\n            raise ValueError(\'invalid request seed\')\\n        new_request = getattr(self, \'_smoke_rid\', None) != rid\\n        if new_request:\\n            seen = getattr(self, \'_smoke_seen\', set())\\n            if rid in seen:\\n                raise ValueError(\'interleaved requests are unsupported by this single-request smoke\')\\n            seen.add(rid)\\n            self._smoke_seen = seen\\n            self._smoke_rid = rid\\n            self._smoke_metadata = (run_id, seed)\\n            self._smoke_step = 0\\n            # Initialize the actual worker RNG once, at the first sample of this request.\\n            torch.random.default_generator.manual_seed(seed)\\n            if output.next_token_logits.is_cuda:\\n                with torch.cuda.device(output.next_token_logits.device):\\n                    torch.cuda.manual_seed(seed)\\n        elif self._smoke_metadata != (run_id, seed):\\n            raise ValueError(\'metadata changed within an active request\')\\n        directory = str(Path(root) / run_id / \'events\')\\n        if output.next_token_logits.shape[0] != 1 or info.latent_modes.numel() != 1:\\n            raise ValueError(\'K0 requires exactly one request row\')\\n        mode = bool(info.latent_modes.item())\\n        record = dict(generation_idx=getattr(self, \'_smoke_step\', 0),\\n                      latent_mode=mode, worker_pid=os.getpid(), seed=torch.initial_seed(),\\n                      request_id=rid, run_id=run_id, request_seed_applied=new_request)\\n        # Read only: no RNG calls, no full-vocabulary copy/history.\\n        if mode:\\n            hidden = output.hidden_states\\n            if hidden is None or hidden.ndim != 2 or hidden.shape[0] != 1:\\n                raise ValueError(\'LAST hidden missing or not aligned to single request\')\\n            scores, ids = torch.topk(output.next_token_logits.float(), k=10, dim=-1)\\n            record.update(hidden_state=hidden[0].detach().float().cpu().tolist(),\\n                          topk_probs=scores.softmax(-1)[0].detach().cpu().tolist(),\\n                          clean_topk_ids=ids[0].detach().cpu().tolist())\\n            del scores, ids, hidden\\n        tokens = original(self, output, info, *args, enable_latent=enable_latent, **kwargs)\\n        token = int(tokens[0].item())\\n        record.update(next_token_id=token,\\n                      is_mixture_step=mode and token != int(os.environ[\'ADANOISE_SMOKE_END_ID\']))\\n        if record[\'is_mixture_step\']:\\n            record.update(topk_ids=output.topk_indices[0].detach().cpu().tolist(),\\n                          mixture_probs=output.topk_probs[0].detach().float().cpu().tolist())\\n        if output.next_token_logits.is_cuda:\\n            device = output.next_token_logits.device\\n            with torch.cuda.device(device):\\n                record[\'cuda_seed\'] = torch.cuda.initial_seed()\\n            record.update(worker_peak_allocated_mb=torch.cuda.max_memory_allocated(device)/2**20,\\n                          worker_peak_reserved_mb=torch.cuda.max_memory_reserved(device)/2**20)\\n        path = Path(directory) / f\'{os.getpid()}.jsonl\'\\n        with path.open(\'a\', encoding=\'utf-8\') as stream:\\n            stream.write(json.dumps(record, allow_nan=False) + \'\\\\n\')\\n        self._smoke_step = record[\'generation_idx\'] + 1\\n        return tokens\\n    sampler_class.forward = captured\\n"\n\ndef sha256_bytes(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\ndef package_hash(package):\n    files = {p.relative_to(package).as_posix(): file_hash(p)\n             for p in sorted(Path(package).rglob(\'*.py\'))}\n    return sha256_bytes(json.dumps(files, sort_keys=True).encode())\n\ndef rollout_seed(rollout_id):\n    payload = json.dumps([12345, PROBLEM_ID, \'1\', rollout_id],\n                         ensure_ascii=True, separators=(\',\', \':\')).encode(\'ascii\')\n    return int.from_bytes(hashlib.sha256(b\'adanoise.probe-seeds.v1:\' + payload).digest()[:4], \'big\')\n\ndef check_probabilities(values):\n    if (len(values) != 10 or any(not math.isfinite(x) or x < 0 for x in values)\n            or not math.isclose(sum(values), 1.0, abs_tol=1e-5)):\n        raise ValueError(\'invalid top10 probabilities\')\n\ndef validate_events(records, output_ids, seed, end_id, expected_hidden_size=None):\n    if not records or [r[\'generation_idx\'] for r in records] != list(range(len(records))):\n        raise ValueError(\'missing or discontinuous sampler events\')\n    if [r[\'next_token_id\'] for r in records] != output_ids:\n        raise ValueError(\'sampler token sequence differs from returned output_ids\')\n    if (any(r[\'seed\'] != seed or (\'cuda_seed\' in r and r[\'cuda_seed\'] != seed) for r in records)\n            or len({r[\'worker_pid\'] for r in records}) != 1):\n        raise ValueError(\'worker seed or single-worker identity mismatch\')\n    if not records[0][\'latent_mode\']:\n        raise ValueError(\'first generation step did not enter latent mode\')\n    steps = []\n    hidden_size = None\n    exited = False\n    for record in records:\n        if record[\'latent_mode\']:\n            if exited:\n                raise ValueError(\'latent mode reentered after explicit generation\')\n            mixture = record[\'next_token_id\'] != end_id\n            if record[\'is_mixture_step\'] != mixture:\n                raise ValueError(\'latent exit and mixture mask mismatch\')\n        else:\n            exited = True\n            if record[\'is_mixture_step\']:\n                raise ValueError(\'explicit token mislabeled as mixture step\')\n        if not record[\'is_mixture_step\']:\n            continue\n        hidden = record[\'hidden_state\']\n        if not hidden or any(not math.isfinite(x) for x in hidden):\n            raise ValueError(\'empty or nonfinite hidden vector\')\n        hidden_size = hidden_size or len(hidden)\n        if len(hidden) != hidden_size or (expected_hidden_size is not None and len(hidden) != expected_hidden_size):\n            raise ValueError(\'hidden dimension changed within trajectory\')\n        check_probabilities(record[\'topk_probs\'])\n        check_probabilities(record[\'mixture_probs\'])\n        for name in (\'clean_topk_ids\', \'topk_ids\'):\n            ids = record[name]\n            if len(ids) != 10 or len(set(ids)) != 10 or any(type(x) is not int or x < 0 for x in ids):\n                raise ValueError(\'invalid sparse top10 token IDs\')\n        if record[\'topk_ids\'][0] != record[\'next_token_id\']:\n            raise ValueError(\'mixture representative token mismatch\')\n        steps.append(record)\n    if not steps:\n        raise ValueError(\'no actual latent mixture steps\')\n    return {\'num_latent_steps\': len(steps), \'hidden_size\': hidden_size,\n            \'latent_exit_observed\': any(r[\'latent_mode\'] and r[\'next_token_id\'] == end_id for r in records)}\n\ndef load_question():\n    import pyarrow.parquet as pq\n    path = Path(DATA_PATH)\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    if file_hash(path) != SOURCE_SHA256:\n        raise ValueError(\'mounted GSM8K file differs from fixed candidate-pool source\')\n    offset = 0\n    for batch in pq.ParquetFile(path).iter_batches(batch_size=2048):\n        if offset <= CANDIDATE_ROW < offset + batch.num_rows:\n            row = batch.slice(CANDIDATE_ROW-offset, 1).to_pylist()[0]\n            if not row.get(\'prompt\') or not row.get(\'reward_model\', {}).get(\'ground_truth\'):\n                raise ValueError(\'selected fixed train candidate lacks prompt/answer\')\n            return row\n        offset += batch.num_rows\n    raise ValueError(\'fixed train candidate row missing\')\n\ndef grade_output(text, truth, scoring_path):\n    names = (\'fix_fracs\', \'fix_a_slash_b\', \'remove_right_units\', \'fix_sqrt\', \'strip_string\',\n             \'remove_boxed\', \'last_boxed_only_string\', \'extract_answer\',\n             \'normalize_answer_text\', \'_as_float\', \'check_is_correct\')\n    tree = ast.parse(Path(scoring_path).read_text())\n    functions = {n.name: n for n in tree.body if isinstance(n, ast.FunctionDef)}\n    if set(names) - functions.keys():\n        raise ValueError(\'required author grading helpers missing\')\n    ns = {\'re\': re}\n    exec(compile(ast.Module(body=[functions[n] for n in names], type_ignores=[]), str(scoring_path), \'exec\'), ns)\n    answer = ns[\'extract_answer\'](text) if text.strip() else \'\'\n    invalid = \'empty_output\' if not text.strip() else (\'answer_unextractable\' if not answer.strip() else None)\n    return {\'reward\': int(ns[\'check_is_correct\'](answer, truth)) if invalid is None else 0,\n            \'extracted_answer\': answer, \'invalid_reason\': invalid}\n\nclass MemoryMonitor:\n    """Sample entire GPU0 memory, including worker allocations; not a true peak."""\n    def __init__(self):\n        self.stop_event = threading.Event()\n        self.samples = []\n        self.error = None\n        self.thread = threading.Thread(target=self._run, daemon=True)\n\n    def _run(self):\n        while not self.stop_event.is_set():\n            try:\n                text = subprocess.check_output(\n                    [\'nvidia-smi\', \'--query-gpu=index,memory.used\', \'--format=csv,noheader,nounits\'],\n                    text=True, timeout=10)\n                value = next(float(line.split(\',\')[1]) for line in text.splitlines()\n                             if int(line.split(\',\')[0]) == 0)\n                self.samples.append(value)\n            except Exception as error:\n                self.error = repr(error)\n                return\n            self.stop_event.wait(0.5)\n\n    def stop(self):\n        self.stop_event.set()\n        self.thread.join()\n\n\ndef prepare_overlay(root):\n    spec = importlib.util.find_spec(\'sglang\')\n    if spec is None or not spec.origin:\n        raise ModuleNotFoundError(\'author custom SGLang is required\')\n    package = Path(spec.origin).resolve().parent\n    relative_paths = (\'srt/layers/sampler.py\', \'srt/sampling/sampling_batch_info.py\')\n    for relative in relative_paths:\n        if not (package / relative).is_file():\n            raise FileNotFoundError(package / relative)\n    scoring = package.parents[2] / \'eval/eval_low_tasks_sglang.py\'\n    if not scoring.is_file():\n        raise FileNotFoundError(scoring)\n    # Record provenance only. No comparison against a pinned commit or source SHA.\n    source = {\'installed_package_path\': str(package), \'package_sha256\': package_hash(package),\n              \'scoring_sha256\': file_hash(scoring)}\n    overlay = root / \'runtime_overlay\'\n    overlay.mkdir()\n    shutil.copytree(package, overlay / \'sglang\', ignore=shutil.ignore_patterns(\'__pycache__\', \'*.pyc\'))\n    (overlay / \'adanoise_smoke_capture.py\').write_text(CAPTURE_SOURCE)\n    (overlay / \'adanoise_request_metadata.py\').write_text(BATCH_SOURCE)\n    with (overlay / \'sglang/srt/layers/sampler.py\').open(\'a\') as stream:\n        stream.write(\'\\nfrom adanoise_smoke_capture import install as _smoke_install\\n_smoke_install(Sampler)\\n\')\n    with (overlay / \'sglang/srt/sampling/sampling_batch_info.py\').open(\'a\') as stream:\n        stream.write(\'\\nfrom adanoise_request_metadata import install_batch as _install_metadata\\n_install_metadata(SamplingBatchInfo)\\n\')\n    return overlay, scoring, source\n\n\ndef summarize_results(results):\n    if len(results) != 3:\n        raise ValueError(\'expected three sequential results\')\n    a, b, c = results\n    if len({r[\'worker_pid\'] for r in results}) != 1:\n        raise ValueError(\'sampling worker changed; persistent Engine not verified\')\n    if len({r[\'request_id\'] for r in results}) != 3:\n        raise ValueError(\'expected three distinct scheduler request IDs\')\n    if a[\'seed\'] != b[\'seed\'] or a[\'seed\'] == c[\'seed\']:\n        raise ValueError(\'request seed identities do not match the repeat/difference design\')\n    same = all(a[key] == b[key] for key in (\'output_ids\', \'trace_hash\', \'hidden_hash\'))\n    if not same:\n        raise ValueError(\'same request seed failed token/mixture/hidden reproducibility\')\n    different = a[\'trace_hash\'] != c[\'trace_hash\']\n    return dict(status=\'passed\' if different else \'inconclusive\', same_seed_passed=same,\n                different_seed_trace_changed=different, persistent_worker_verified=True,\n                engine_start_count=1, num_problems=1, num_requests=3,\n                scope=\'single_gpu_single_active_request_persistent_engine_seed\')\n\n\ndef collect_result(root, run_id, seed, output, tokenizer, context, end_id, generation_sec, sampled_memory):\n    run_dir = root / run_id\n    paths = list((run_dir / \'events\').glob(\'*.jsonl\'))\n    if len(paths) != 1:\n        raise ValueError(\'expected one sampling worker event stream\')\n    records = [json.loads(line) for line in paths[0].read_text().splitlines()]\n    if not records or any(\'cuda_seed\' not in r for r in records):\n        raise ValueError(\'CUDA sampling worker seed observation missing\')\n    if (not records[0][\'request_seed_applied\']\n            or any(r[\'request_seed_applied\'] for r in records[1:])\n            or len({r[\'request_id\'] for r in records}) != 1\n            or any(r[\'run_id\'] != run_id for r in records)):\n        raise ValueError(\'request identity or first-step-only seed initialization mismatch\')\n    request_id = records[0][\'request_id\']\n    returned_id = output.get(\'meta_info\', {}).get(\'id\')\n    if returned_id is not None and returned_id != request_id:\n        raise ValueError(\'returned scheduler request ID differs from sampler request\')\n    ids = output[\'output_ids\']\n    checks = validate_events(records, ids, seed, end_id, context[\'expected_hidden_size\'])\n    decoded = tokenizer.decode(ids, skip_special_tokens=False)\n    score = grade_output(decoded, context[\'question\'][\'reward_model\'][\'ground_truth\'], context[\'scoring_path\'])\n    finish = output.get(\'meta_info\', {}).get(\'finish_reason\')\n    reason = finish.get(\'type\') if isinstance(finish, dict) else finish\n    identity = dict(schema_version=1, problem_id=PROBLEM_ID, scale=1.0,\n                    rollout_id=0 if run_id != \'different\' else 1, seed=seed,\n                    model_identity=context[\'model_identity\'], generation_config=EFFECTIVE_GENERATION_CONFIG)\n    mixed = [r for r in records if r[\'is_mixture_step\']]\n    features = dict(identity, feature_schema=\'raw-latent-v1\', feature_source=\'pre_gumbel\',\n                    steps=[dict(step_idx=i, hidden_state=r[\'hidden_state\'], topk_probs=r[\'topk_probs\'])\n                           for i, r in enumerate(mixed)])\n    trajectory = dict(identity, output_text=decoded, all_output_ids=ids, reward=score[\'reward\'],\n                      explicit_token_ids=[r[\'next_token_id\'] for r in records if not r[\'latent_mode\']],\n                      steps=[dict(step_idx=i, topk_ids=r[\'topk_ids\'], mixture_probs=r[\'mixture_probs\'])\n                             for i, r in enumerate(mixed)])\n    write_json(run_dir / \'features.json\', features)\n    write_json(run_dir / \'trajectory.json\', trajectory)\n    stochastic = [{k: r[k] for k in (\'next_token_id\', \'latent_mode\', \'is_mixture_step\',\n                                   \'topk_ids\', \'mixture_probs\') if k in r} for r in records]\n    result = dict(identity, status=\'passed\', **checks, **score, request_id=request_id,\n                  worker_pid=records[0][\'worker_pid\'], output_text=decoded, output_ids=ids,\n                  output_hash=sha256_bytes(decoded.encode()),\n                  trace_hash=sha256_bytes(json.dumps(stochastic, sort_keys=True).encode()),\n                  hidden_hash=sha256_bytes(json.dumps(features[\'steps\'], sort_keys=True).encode()),\n                  finish_reason=finish, truncated=reason in (\'length\', \'max_tokens\') if reason else None,\n                  budget_hit=len(ids) >= GENERATION_CONFIG[\'max_new_tokens\'],\n                  instrumented_generation_sec=generation_sec, sampled_device_memory_max_mb=sampled_memory,\n                  worker_peak_allocated_mb=max(r.get(\'worker_peak_allocated_mb\', 0) for r in records),\n                  worker_peak_reserved_mb=max(r.get(\'worker_peak_reserved_mb\', 0) for r in records),\n                  memory_scope=\'peaks_since_engine_start_and_device_samples\',\n                  request_seed_initialized_once=True, prefix_cache_enabled=False)\n    write_json(run_dir / \'result.json\', result)\n    return result\n\n\ndef run_persistent(root):\n    import asyncio\n    import torch\n    import sglang as sgl\n    from transformers import AutoTokenizer\n    context = json.loads((root / \'input.json\').read_text())\n    os.environ[\'ADANOISE_SMOKE_ROOT\'] = str(root)\n    if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != \'Tesla T4\':\n        raise RuntimeError(\'confirmed T4 GPU0 is not available\')\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)\n    text = tokenizer.apply_chat_template(context[\'question\'][\'prompt\'], tokenize=False, add_generation_prompt=True)\n    if not text.rstrip().endswith(\'<think>\'):\n        text += \'<think>\'\n    input_ids = tokenizer.encode(text, add_special_tokens=False)\n    end_ids = tokenizer.encode(\'</think>\', add_special_tokens=False)\n    if not end_ids:\n        raise ValueError(\'empty latent end marker tokenization\')\n    end_id = end_ids[0]\n    os.environ[\'ADANOISE_SMOKE_END_ID\'] = str(end_id)\n    write_json(root / \'prompt.json\', dict(text=text, input_ids=input_ids, end_marker_ids=end_ids,\n                                        latent_end_token_id=end_id))\n    runs = [(\'same_a\', rollout_seed(0)), (\'same_b\', rollout_seed(0)), (\'different\', rollout_seed(1))]\n    for run_id, _ in runs:\n        (root / run_id / \'events\').mkdir(parents=True)\n    monitor = MemoryMonitor()\n    monitor.thread.start()\n    engine = None\n    loop = None\n    start = time.perf_counter()\n    try:\n        engine = sgl.Engine(model_path=MODEL_PATH, trust_remote_code=True, dtype=\'float16\',\n                            kv_cache_dtype=\'auto\', tp_size=1, base_gpu_id=0,\n                            random_seed=ENGINE_STARTUP_SEED, enable_latent=True,\n                            latent_end_token_id=end_id, disable_cuda_graph=True,\n                            disable_overlap_schedule=True, disable_radix_cache=True,\n                            mem_fraction_static=0.90, sampling_backend=\'flashinfer\',\n                            max_running_requests=1, log_level=\'info\', skip_tokenizer_init=True, max_topk=10)\n        loop = asyncio.new_event_loop()\n        asyncio.set_event_loop(loop)\n        startup_sec = time.perf_counter() - start\n        results = []\n        for run_id, seed in runs:\n            print(f\'Persistent request={run_id}, seed={seed}, engine_start_count=1\', flush=True)\n            sample_start = len(monitor.samples)\n            begin = time.perf_counter()\n            params = dict(GENERATION_CONFIG, custom_params={\n                \'adanoise_smoke\': dict(run_id=run_id, seed=seed)})\n            output = engine.generate(input_ids=input_ids, sampling_params=params)\n            generation_sec = time.perf_counter() - begin\n            if monitor.error or not monitor.samples:\n                raise RuntimeError(f\'GPU memory measurement failed: {monitor.error}\')\n            values = monitor.samples[sample_start:]\n            # A request can finish between two device samples; do not invent a per-request peak.\n            sampled_memory = max(values) if values else None\n            result = collect_result(root, run_id, seed, output, tokenizer, context, end_id,\n                                    generation_sec, sampled_memory)\n            results.append(result)\n            print(json.dumps(result, ensure_ascii=False), flush=True)\n        checks = summarize_results(results)\n        monitor.stop()\n        if monitor.error:\n            raise RuntimeError(f\'GPU memory measurement failed: {monitor.error}\')\n        summary = dict(checks, problem_id=PROBLEM_ID, model_identity=context[\'model_identity\'],\n                       startup_sec=startup_sec, engine_startup_seed=ENGINE_STARTUP_SEED,\n                       prefix_cache_enabled=False, max_running_requests=1,\n                       sampled_device_memory_max_mb=max(monitor.samples), memory_sample_interval_sec=0.5,\n                       gpu_name=torch.cuda.get_device_name(0), torch=torch.__version__, sglang=sgl.__version__,\n                       runs=[dict(run_id=name, **r) for (name, _), r in zip(runs, results)],\n                       pending=[\'multi_request_concurrent_rng_isolation\', \'formal_probing_throughput_budget\'])\n        write_json(root / \'smoke_test.json\', summary)\n        print(f\'REQUEST SEED {checks["status"].upper()}: {root / "smoke_test.json"}\', flush=True)\n        if checks[\'status\'] != \'passed\':\n            raise ValueError(\'different request seed did not change trace; inspection required\')\n    finally:\n        monitor.stop()\n        if engine is not None:\n            engine.shutdown()\n        if loop is not None:\n            loop.close()\n            asyncio.set_event_loop(None)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output-dir\')\n    parser.add_argument(\'--worker\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.worker:\n        run_persistent(Path(args.output_dir))\n        return\n    root = Path(args.output_dir) if args.output_dir else Path(\'/kaggle/working/artifacts\') / (\'request_seed_smoke_\' + time.strftime(\'%Y%m%d_%H%M%S\'))\n    if root.exists():\n        raise FileExistsError(f\'refusing to overwrite output: {root}\')\n    if not Path(MODEL_PATH).is_dir():\n        raise FileNotFoundError(MODEL_PATH)\n    root.mkdir(parents=True)\n    write_json(root / \'smoke_test.json\', dict(status=\'incomplete\', scope=\'persistent_request_seed\'))\n    try:\n        question = load_question()\n        overlay, scoring, provenance = prepare_overlay(root)\n        model = Path(MODEL_PATH)\n        weights = sorted(model.glob(\'*.safetensors\'))\n        if not weights or not (model / \'config.json\').is_file():\n            raise FileNotFoundError(\'GRPO checkpoint config/weights missing\')\n        files = [model / name for name in (\'config.json\', \'tokenizer.json\', \'tokenizer_config.json\',\n                                          \'special_tokens_map.json\') if (model / name).is_file()] + weights\n        model_hashes = {p.name: file_hash(p) for p in files}\n        hidden_size = json.loads((model / \'config.json\').read_text())[\'hidden_size\']\n        if type(hidden_size) is not int or hidden_size <= 0:\n            raise ValueError(\'model config requires a positive hidden_size\')\n        write_json(root / \'input.json\', dict(question=question, problem_id=PROBLEM_ID, split=\'train\',\n                   model_path=MODEL_PATH, model_identity=sha256_bytes(json.dumps(model_hashes, sort_keys=True).encode()),\n                   model_file_sha256=model_hashes, data_sha256=SOURCE_SHA256,\n                   expected_hidden_size=hidden_size, scoring_path=str(scoring), runtime_provenance=provenance,\n                   capture_hook_sha256=sha256_bytes(CAPTURE_SOURCE.encode()),\n                   metadata_hook_sha256=sha256_bytes(BATCH_SOURCE.encode())))\n        env = os.environ.copy()\n        env[\'PYTHONPATH\'] = str(overlay) + os.pathsep + env.get(\'PYTHONPATH\', \'\')\n        subprocess.run([sys.executable, str(Path(__file__).resolve()), \'--worker\', \'--output-dir\', str(root)],\n                       env=env, check=True)\n    except Exception as error:\n        write_json(root / \'failure.json\', dict(status=\'failed\', error_type=type(error).__name__, error=str(error)))\n        raise  # STOP, no environment changes, retries or automatic formal probing\n\n\nif __name__ == \'__main__\':\n    main()\n', 'configs/kaggle_candidate_rows.json': '{\n  "schema_version": 1,\n  "master_seed": 42,\n  "original_manifest_sha256": "dd3bf185cf363fe5c98867f3d040d2e3719f4c58c1268a242544d619554fe85c",\n  "inputs": {\n    "gsm8k_aug": {\n      "sha256": "3766e3a83cd82ddd686392d8bc6ef6f262821490a09b694b2caed44f1a482501"\n    },\n    "dapo_math": {\n      "sha256": "6f0bc38023dc691533841d37f31a267acd3f277b60fc8746cc85b6493aa04726"\n    }\n  },\n  "rows": [\n    {\n      "source": "gsm8k_aug",\n      "source_row": 1611,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 3297,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 5039,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 7335,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 9872,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 18670,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 18783,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 25690,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 28469,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 28993,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 31240,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 32611,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 37803,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 40891,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 40955,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 42143,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 43556,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 50020,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 52757,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 56565,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 57677,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 58960,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 64367,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 65496,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 79757,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 80931,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 82634,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 85733,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 94516,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 106047,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 108080,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 112488,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 116492,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 119826,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 120907,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 121800,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 124289,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 130890,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 133743,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 134404,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 140242,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 141155,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 144217,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 148168,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 150030,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 154310,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 155514,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 157511,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 157793,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 162700,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 166130,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 173491,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 173751,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 173992,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 175901,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 178011,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 185127,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 189040,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 192916,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 193642,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 194553,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 196168,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 196381,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 199708,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 201254,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 201780,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 202897,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 203033,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 203372,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 209608,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 213073,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 213512,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 218808,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 221788,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 221873,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 223529,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 225471,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 225940,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 233637,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 233855,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 234099,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 235032,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 240934,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 246309,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 246369,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 247868,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 248782,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 251017,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 251065,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 252184,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 254296,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 255458,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 257561,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 257668,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 258277,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 262247,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 265216,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 265985,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 267790,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 268379,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 271194,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 276852,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 277786,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 278513,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 279364,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 282323,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 283288,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 287578,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 290760,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 291233,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 291902,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 292504,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 293064,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 294548,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 295315,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 295484,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 295675,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 296841,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 298535,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 300514,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 237,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 384,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 453,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 459,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 556,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 573,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 622,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 860,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1042,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1131,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1159,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1558,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1564,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1606,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1683,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2145,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2185,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2190,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2291,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2419,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2432,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2491,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2721,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2942,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2972,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3125,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3202,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3302,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3401,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3486,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3710,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3769,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3859,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4052,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4123,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4277,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4712,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5088,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5139,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5315,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5425,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5507,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5629,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5653,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5736,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5956,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6109,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6111,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6436,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6807,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6838,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6848,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6977,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7013,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7132,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7229,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7340,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7356,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7473,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7703,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7988,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7995,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8322,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8327,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8352,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8362,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8402,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8466,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8912,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8963,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9028,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9177,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9243,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9247,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9309,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9342,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9450,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9550,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 9894,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10012,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10078,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10180,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10183,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10209,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10266,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10335,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10371,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10575,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10585,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10681,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10820,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10908,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10939,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11018,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11098,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11255,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11476,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11550,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11579,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11598,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11719,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11726,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11871,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11961,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12049,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12082,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12167,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12256,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12286,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12287,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12403,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12671,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12691,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12857,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12902,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12977,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13504,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13547,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13803,\n      "split": "train"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 14108,\n      "split": "train"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 4230,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 23828,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 29268,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 34033,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 38145,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 39657,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 46498,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 48722,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 49148,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 58296,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 84238,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 87288,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 97420,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 97752,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 111939,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 116234,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 119885,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 129546,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 135867,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 141812,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 147924,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 169403,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 171540,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 183401,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 184563,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 192477,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 206891,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 210286,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 215258,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 217687,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 219839,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 251020,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 258318,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 270012,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 273316,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 273433,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 279689,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 281114,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 298744,\n      "split": "validation"\n    },\n    {\n      "source": "gsm8k_aug",\n      "source_row": 299202,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 269,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 551,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 1030,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2386,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 2539,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3280,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 3372,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4259,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4314,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4385,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4452,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4482,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 4568,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5271,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5323,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5366,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5691,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 5862,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6308,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6327,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 6446,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7582,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 7922,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8215,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8765,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 8974,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10043,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 10972,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11113,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11176,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11223,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11353,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 11781,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12011,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12148,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12606,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 12901,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13323,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13716,\n      "split": "validation"\n    },\n    {\n      "source": "dapo_math",\n      "source_row": 13778,\n      "split": "validation"\n    }\n  ]\n}\n'}
root = Path('/kaggle/working/adanoise_probe_bundle')
for relative, content in files.items():
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
print('Formal probing bundle READY')
BUNDLE

cat > /kaggle/working/test_sglang_seed.py <<'PY'
"""Approved Kaggle frozen-checkpoint probing, single persistent Engine, <=8h/run."""
import argparse
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import uuid

ENTRY_STARTED=time.monotonic()

ROOT=Path(os.environ.get('ADANOISE_PROJECT_ROOT',Path(__file__).resolve().parents[1]))
sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'scripts'))
from adaptive_noise.kaggle_probing import ProbeStore, atomic_json, request_key, restore_candidates
from adaptive_noise.probing import generation_settings
from adaptive_noise.answer_verifier import load_author_helpers
import kaggle_request_seed_smoke as smoke

MAX_SECONDS=8*3600
SHUTDOWN_MARGIN_SECONDS=300
MODEL_PATH=smoke.MODEL_PATH
DATA_ROOT=Path('/kaggle/input/datasets/fengjixing/datasets-for-latent-grpo/data')
FILES={'gsm8k_aug':'GSM8k-Aug-oss-dup-all.parquet','dapo_math':'DAPO-Math-17k-en-train.parquet'}
# Same validated hook, generalized only from three diagnostic names to private attempt IDs.
CAPTURE_SOURCE=smoke.CAPTURE_SOURCE.replace(
    "run_id not in ('same_a', 'same_b', 'different')",
    "not isinstance(run_id, str) or len(run_id) != 32 or any(c not in '0123456789abcdef' for c in run_id)")


def grade(source,text,truth,path):
    profile='low' if source=='gsm8k_aug' else 'high'
    helpers=load_author_helpers(profile,path)
    answer=''
    if text.strip():
        if profile=='low':
            answer=helpers['extract_answer'](text)
        else:
            boxed=helpers['last_boxed_only_string'](text)
            if boxed is not None:
                try:
                    answer=helpers['remove_boxed'](boxed)
                except AssertionError:
                    answer=''
    invalid='empty_output' if not text.strip() else ('answer_unextractable' if not answer.strip() else None)
    return dict(reward=int(helpers['check_is_correct'](answer,truth) if profile=='low' else helpers['is_equiv'](answer,truth)) if invalid is None else 0,
                invalid_reason=invalid,extracted_answer=answer)


def collect_pending(store,sample,deadline,clock=time.monotonic):
    try:
        for candidate in store.mapping.values():
            while True:
                pending=store.pending(candidate)
                if not pending:
                    store.finish_questions()
                    break
                for request in pending:
                    if clock()>=deadline:
                        store.snapshot('budget_stopped')
                        return 'budget_stopped'
                    attempt=store.new_attempt()
                    row=sample(candidate,request,attempt)
                    saved=store.commit(row,attempt)
                    print(json.dumps({k:saved.get(k) for k in ('problem_id','split','scale','rollout_id','seed','reward','invalid_reason','instrumented_generation_sec')})+f' completed={len(store.history)}',flush=True)
            store.snapshot('running')
        store.publish()
        return 'complete'
    except Exception:
        store.snapshot('failed')
        raise  # no automatic retries or conversion of engine errors to reward=0


def stop_group(process):
    try:
        os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        pass
    # Leader exit alone does not guarantee all Engine workers exited.
    try:
        os.killpg(process.pid,signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=10)


def supervise(command,env,timeout):
    if timeout<=0:
        return 'budget_stopped'
    process=subprocess.Popen(command,env=env,start_new_session=True)
    try:
        code=process.wait(timeout=timeout)
        if code:
            stop_group(process)
            raise subprocess.CalledProcessError(code,command)
        return 'finished'
    except subprocess.TimeoutExpired:
        stop_group(process)
        return 'budget_stopped'
    except BaseException:
        stop_group(process)
        raise


def collect_output(attempt,candidate,request,output,tokenizer,context,end_id,elapsed):
    paths=list((attempt/'events').glob('*.jsonl'))
    if len(paths)!=1:
        raise ValueError('expected one CUDA worker event stream')
    records=[json.loads(line) for line in paths[0].read_text().splitlines()]
    if not records or any(r.get('cuda_seed')!=request['seed'] for r in records):
        raise ValueError('missing/mismatched CUDA request seed')
    if not records[0]['request_seed_applied'] or any(r['request_seed_applied'] for r in records[1:]):
        raise ValueError('request seed must be initialized once')
    if len({r['request_id'] for r in records})!=1 or any(r['run_id']!=attempt.name for r in records):
        raise ValueError('sampler request metadata mismatch')
    rid=records[0]['request_id']
    returned=output.get('meta_info',{}).get('id')
    if returned is not None and returned!=rid:
        raise ValueError('returned request ID differs from sampler')
    ids=output['output_ids']
    checks=smoke.validate_events(records,ids,request['seed'],end_id,context['hidden_size'])
    text=tokenizer.decode(ids,skip_special_tokens=False)
    score=grade(candidate['source'],text,candidate['ground_truth'],context['scoring_paths'][candidate['source']])
    finish=output.get('meta_info',{}).get('finish_reason')
    reason=finish.get('type') if isinstance(finish,dict) else finish
    identity=dict(schema_version=1,**{k:request[k] for k in ('problem_id','scale','rollout_id','seed')},
                  model_identity=context['model_identity'],generation_config=generation_settings(candidate['source'],request['scale']))
    mixed=[r for r in records if r['is_mixture_step']]
    features=dict(identity,feature_schema='raw-latent-v1',feature_source='pre_gumbel',
        steps=[dict(step_idx=i,hidden_state=r['hidden_state'],topk_probs=r['topk_probs']) for i,r in enumerate(mixed)])
    trajectory=dict(identity,output_text=text,reward=score['reward'],all_output_ids=ids,
        explicit_token_ids=[r['next_token_id'] for r in records if not r['latent_mode']],
        steps=[dict(step_idx=i,topk_ids=r['topk_ids'],mixture_probs=r['mixture_probs']) for i,r in enumerate(mixed)])
    atomic_json(attempt/'features.json',features)
    atomic_json(attempt/'trajectory.json',trajectory)
    return dict(identity,source=candidate['source'],split=candidate['split'],**score,**checks,
        request_id=rid,worker_pid=records[0]['worker_pid'],request_seed_initialized_once=True,
        finish_reason=reason,truncated=reason in ('length','max_tokens') if reason else None,
        budget_hit=len(ids)>=request['max_new_tokens'],instrumented_generation_sec=elapsed,
        worker_peak_allocated_mb=max(r.get('worker_peak_allocated_mb',0) for r in records),
        worker_peak_reserved_mb=max(r.get('worker_peak_reserved_mb',0) for r in records),
        memory_scope='peaks_since_engine_start')


def run_worker(root,session,deadline):
    import asyncio
    import torch
    import sglang as sgl
    from transformers import AutoTokenizer
    context=json.loads((session/'input.json').read_text())
    store=ProbeStore(root,root/'candidates',context['model_identity'])
    if time.monotonic()>=deadline:
        store.snapshot('budget_stopped')
        return
    if not torch.cuda.is_available() or torch.cuda.get_device_name(0)!='Tesla T4':
        raise RuntimeError('confirmed T4 GPU0 unavailable')
    tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=True)
    end_ids=tokenizer.encode('</think>',add_special_tokens=False)
    if not end_ids:
        raise ValueError('missing latent end marker')
    end_id=end_ids[0]
    os.environ['ADANOISE_SMOKE_ROOT']=str(root/'attempts')
    os.environ['ADANOISE_SMOKE_END_ID']=str(end_id)
    engine=None
    loop=None
    monitor=smoke.MemoryMonitor()
    monitor.thread.start()
    worker_pid=None
    seen_rids=set()
    start=time.monotonic()
    try:
        engine=sgl.Engine(model_path=MODEL_PATH,trust_remote_code=True,dtype='float16',kv_cache_dtype='auto',
            tp_size=1,base_gpu_id=0,random_seed=12345,enable_latent=True,latent_end_token_id=end_id,
            disable_cuda_graph=True,disable_overlap_schedule=True,disable_radix_cache=True,
            mem_fraction_static=.90,sampling_backend='flashinfer',max_running_requests=1,
            log_level='info',skip_tokenizer_init=True,max_topk=10)
        loop=asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        startup=time.monotonic()-start
        def sample(candidate,request,attempt):
            nonlocal worker_pid
            text=tokenizer.apply_chat_template(candidate['prompt'],tokenize=False,add_generation_prompt=True)
            if not text.rstrip().endswith('<think>'):
                text+='<think>'
            ids=tokenizer.encode(text,add_special_tokens=False)
            atomic_json(attempt/'prompt.json',dict(text=text,input_ids=ids,end_marker_ids=end_ids))
            params={k:v for k,v in generation_settings(candidate['source'],request['scale']).items() if k!='max_topk'}
            params['custom_params']={'adanoise_smoke':dict(run_id=attempt.name,seed=request['seed'])}
            begin=time.monotonic()
            output=engine.generate(input_ids=ids,sampling_params=params)
            elapsed=time.monotonic()-begin
            if monitor.error:
                raise RuntimeError(f'GPU measurement failed: {monitor.error}')
            row=collect_output(attempt,candidate,request,output,tokenizer,context,end_id,elapsed)
            if worker_pid is not None and worker_pid!=row['worker_pid']:
                raise ValueError('persistent Engine sampling worker changed')
            if row['request_id'] in seen_rids:
                raise ValueError('scheduler reused request ID')
            worker_pid=row['worker_pid']; seen_rids.add(row['request_id'])
            return row
        status=collect_pending(store,sample,deadline)
        monitor.stop()
        if monitor.error or not monitor.samples:
            raise RuntimeError(f'GPU measurement failed: {monitor.error}')
        atomic_json(session/'result.json',dict(status=status,engine_start_count=1,worker_pid=worker_pid,
            startup_sec=startup,completed_rollouts=len(store.history),elapsed_worker_sec=time.monotonic()-start,
            sampled_device_memory_max_mb=max(monitor.samples),memory_sample_interval_sec=.5,
            prefix_cache_enabled=False,max_running_requests=1,gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__,sglang=sgl.__version__))
        print(f'PROBING {status.upper()}: {root}/collection_status.json',flush=True)
    finally:
        monitor.stop()
        if engine is not None:
            engine.shutdown()
        if loop is not None:
            loop.close()
            asyncio.set_event_loop(None)


def main():
    began=ENTRY_STARTED
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',type=Path,default=Path('/kaggle/working/artifacts/probing_v1'))
    parser.add_argument('--pool',type=Path,default=ROOT/'configs/kaggle_candidate_rows.json')
    parser.add_argument('--data-root',type=Path,default=DATA_ROOT)
    parser.add_argument('--hours',type=float,default=8.)
    parser.add_argument('--prepare',action='store_true')
    parser.add_argument('--began',type=float)
    parser.add_argument('--worker',action='store_true')
    parser.add_argument('--session',type=Path)
    parser.add_argument('--deadline',type=float)
    args=parser.parse_args()
    if args.worker:
        run_worker(args.output_dir,args.session,args.deadline)
        return 0
    if not 0<args.hours<=8 or args.hours*3600<=SHUTDOWN_MARGIN_SECONDS+30:
        parser.error('--hours must be >330 seconds and <=8 hours')
    root=args.output_dir.resolve()
    budget=args.hours*3600
    if not args.prepare:
        command=[sys.executable,str(Path(__file__).resolve()),'--prepare','--began',str(began),
            '--output-dir',str(root),'--pool',str(args.pool.resolve()),'--data-root',str(args.data_root),
            '--hours',str(args.hours)]
        try:
            status=supervise(command,os.environ.copy(),timeout=began+budget-30-time.monotonic())
            if status=='budget_stopped':
                root.mkdir(parents=True,exist_ok=True)
                atomic_json(root/'watchdog.json',dict(status=status,total_elapsed_sec=time.monotonic()-began,
                    note='Preparation and generation supervised together; committed rollouts survive.'))
                print(f'PROBING BUDGET_STOPPED: resume with the same --output-dir {root}',flush=True)
            return 0
        except subprocess.CalledProcessError as error:
            print(f'STOP: preparation/Engine process exited {error.returncode}; see failure.json',file=sys.stderr)
            return 2
    if args.began is None:
        parser.error('--prepare requires --began from the supervising parent')
    began=args.began
    session=root/'sessions'/uuid.uuid4().hex
    store=None
    try:
        if not Path(MODEL_PATH).is_dir():
            raise FileNotFoundError(MODEL_PATH)
        locked=json.loads(args.pool.read_text())
        root.mkdir(parents=True,exist_ok=True)
        pool_path=root/'locked_pool.json'
        if pool_path.exists() and json.loads(pool_path.read_text())!=locked:
            raise ValueError('locked pool changed; cannot resume this collection')
        atomic_json(pool_path,locked)
        if not (root/'candidates').exists():
            restore_candidates(locked,{s:args.data_root/name for s,name in FILES.items()},root/'candidates')
        else:
            # Resume still checks mounted source bytes, without rerunning sampling/splitting.
            for source,name in FILES.items():
                if smoke.file_hash(args.data_root/name)!=locked['inputs'][source]['sha256']:
                    raise ValueError(f'input identity mismatch: {source}')
        session.mkdir(parents=True)
        overlay,low,provenance=smoke.prepare_overlay(session)
        (overlay/'adanoise_smoke_capture.py').write_text(CAPTURE_SOURCE)
        high=low.with_name('eval_high_tasks_sglang.py')
        for profile,path in [('low',low),('high',high)]:
            load_author_helpers(profile,path)  # required scorer preflight, no fixed commit checks
        model=Path(MODEL_PATH)
        weights=sorted(model.glob('*.safetensors'))
        if not weights or not (model/'config.json').is_file():
            raise FileNotFoundError('checkpoint config/weights missing')
        files=[model/name for name in ('config.json','tokenizer.json','tokenizer_config.json','special_tokens_map.json') if (model/name).is_file()]+weights
        hashes={p.name:smoke.file_hash(p) for p in files}
        hidden_size=json.loads((model/'config.json').read_text())['hidden_size']
        if type(hidden_size) is not int or hidden_size<=0:
            raise ValueError('positive hidden_size required')
        identity=smoke.sha256_bytes(json.dumps(hashes,sort_keys=True).encode())
        context=dict(model_identity=identity,model_file_sha256=hashes,hidden_size=hidden_size,
            scoring_paths={'gsm8k_aug':str(low),'dapo_math':str(high)},runtime_provenance=provenance,
            high_scoring_sha256=smoke.file_hash(high),capture_hook_sha256=smoke.sha256_bytes(CAPTURE_SOURCE.encode()),
            total_budget_seconds=budget,shutdown_margin_seconds=SHUTDOWN_MARGIN_SECONDS,
            started_unix_time=time.time(),input_data_root=str(args.data_root))
        atomic_json(session/'input.json',context)
        store=ProbeStore(root,root/'candidates',identity)
        if all(not store.pending(c) for c in store.mapping.values()):
            store.publish()
            print(f'PROBING COMPLETE (already collected): {root}/selected',flush=True)
            return 0
        # Preparation and all Engine workers stay in this parent's supervised process group.
        sys.path.insert(0,str(overlay))
        os.environ['PYTHONPATH']=str(overlay)+os.pathsep+str(ROOT)+os.pathsep+os.environ.get('PYTHONPATH','')
        run_worker(root,session,began+budget-SHUTDOWN_MARGIN_SECONDS)
        return 0
    except Exception as error:
        if store is not None:
            try:
                store=ProbeStore(root,root/'candidates',store.model_identity)
                store.snapshot('failed')
            except Exception:
                pass  # original runtime error remains the STOP reason
        root.mkdir(parents=True,exist_ok=True)
        atomic_json(root/'failure.json',dict(status='failed',error_type=type(error).__name__,error=str(error)))
        print(f'STOP: {type(error).__name__}: {error}',file=sys.stderr)
        return 2


if __name__=='__main__':
    raise SystemExit(main())
PY

ADANOISE_PROJECT_ROOT=/kaggle/working/adanoise_probe_bundle PYTHONUNBUFFERED=1 \
/kaggle/working/latent_sglang/bin/python \
    /kaggle/working/test_sglang_seed.py --output-dir /kaggle/working/artifacts/probing_v1 --hours 8


In [ ]:
%%bash

if [ ! -f /kaggle/working/artifacts/probing_v1/collection_identity.json ]; then
    echo 'STOP: restore the complete probing_v1 collection before resuming' >&2
    exit 2
fi

ADANOISE_PROJECT_ROOT=/kaggle/working/adanoise_probe_bundle PYTHONUNBUFFERED=1 \
/kaggle/working/latent_sglang/bin/python \
    /kaggle/working/test_sglang_seed.py --output-dir /kaggle/working/artifacts/probing_v1 --hours 8
